In [7]:
import pandas as pd
import ast


# 1. Read input files
#    All MusicOSet metadata files are tab-separated

songs = pd.read_csv("musicoset_metadata/songs.csv", sep="\t", low_memory=False)
albums = pd.read_csv("musicoset_metadata/albums.csv", sep="\t", low_memory=False)
artists = pd.read_csv("musicoset_metadata/artists.csv", sep="\t", low_memory=False)
releases = pd.read_csv("musicoset_metadata/releases.csv", sep="\t", low_memory=False)
tracks = pd.read_csv("musicoset_metadata/tracks.csv", sep="\t", low_memory=False)


# 2. Standardize column names

for df in [songs, albums, artists, releases, tracks]:
    df.columns = df.columns.str.strip().str.lower()


# 3. Parse artist_id and artist_name from songs.csv
#    The "artists" column looks like:
#    "{'artist_id': 'artist_name'}"
#    If there are multiple artists, this function keeps the first one

songs_sub = songs[["song_id", "song_name", "artists"]].copy()

def extract_artist_info(x):
    try:
        if pd.isna(x):
            return pd.Series([None, None])
        d = ast.literal_eval(x)
        if isinstance(d, dict) and len(d) > 0:
            artist_id = list(d.keys())[0]
            artist_name = list(d.values())[0]
            return pd.Series([artist_id, artist_name])
        return pd.Series([None, None])
    except Exception:
        return pd.Series([None, None])

songs_sub[["artist_id", "artist_name"]] = songs_sub["artists"].apply(extract_artist_info)
songs_sub = songs_sub.drop(columns=["artists"])


# 4. Keep only needed columns from albums and tracks

albums_sub = albums[["album_id", "name"]].rename(
    columns={"name": "album_name"}
).copy()

tracks_sub = tracks[[
    "song_id",
    "album_id",
    "track_number",
    "release_date",
    "release_date_precision"
]].copy()


# 5. Build the base table
#    tracks links song_id and album_id
#    songs provides song_name + artist_id + artist_name
#    albums provides album_name

base = tracks_sub.merge(songs_sub, on="song_id", how="left")
base = base.merge(albums_sub, on="album_id", how="left")


# 6. Keep only day-level release precision

base = base[base["release_date_precision"] == "day"].copy()


# 7. Convert release_date to datetime
#    Then keep only releases from 2008 to 2018

base["release_date"] = pd.to_datetime(base["release_date"], errors="coerce")
base = base.dropna(subset=["release_date"])

base = base[
    (base["release_date"] >= pd.Timestamp("2008-01-01")) &
    (base["release_date"] <= pd.Timestamp("2018-12-31"))
].copy()


# 8. Sort records so we can keep the earliest release per song_id
#    If the same song appears in multiple albums, singles, or compilations,
#    keep the earliest day-level release record

base = base.sort_values(
    by=["song_id", "release_date", "track_number"],
    ascending=[True, True, True]
)

base_song_level = base.drop_duplicates(subset=["song_id"], keep="first").copy()


# 9. Keep only the final columns required

final_base_table = base_song_level[[
    "song_id",
    "song_name",
    "artist_id",
    "artist_name",
    "album_id",
    "album_name",
    "release_date",
    "release_date_precision"
]].copy()


# 10. Sort for readability

final_base_table = final_base_table.sort_values(
    by=["release_date", "artist_name", "song_name"]
).reset_index(drop=True)


# 11. Save output

final_base_table.to_csv(
    "base_song_table_2008_2018_day_precision.csv",
    index=False
)


# 12. Basic quality checks

print("Final table shape:", final_base_table.shape)
print("\nFirst 10 rows:")
print(final_base_table.head(10))

print("\nNumber of unique song_id:", final_base_table["song_id"].nunique())
print("Missing song_name:", final_base_table["song_name"].isna().sum())
print("Missing artist_id:", final_base_table["artist_id"].isna().sum())
print("Missing artist_name:", final_base_table["artist_name"].isna().sum())
print("Missing album_id:", final_base_table["album_id"].isna().sum())
print("Missing album_name:", final_base_table["album_name"].isna().sum())

print("\nRelease date precision distribution:")
print(final_base_table["release_date_precision"].value_counts(dropna=False))

print("\nRelease year distribution:")
print(final_base_table["release_date"].dt.year.value_counts().sort_index())

Final table shape: (6593, 8)

First 10 rows:
                  song_id              song_name               artist_id  \
0  43Tzmi4Lw0GcZRAE4xqwOg        Citizen/Soldier  2RTUTCvo6onsAnheUk3aL9   
1  3DelzIn0lPwX4PeOJk9zDF       It's Not My Time  2RTUTCvo6onsAnheUk3aL9   
2  4lyphUu7lt4KKuf8jZKjjh                 Get Up  3q7HBObVc0L8jNeTe5Gofh   
3  45EaIq9lKyoQxxTjFkiGbl  Does Your Mother Know  0LcJLqbBmaGUft1e9Mm8HV   
4  29FNeqjOV2kPWGS55qhtGB    Money, Money, Money  0LcJLqbBmaGUft1e9Mm8HV   
5  5pMmWfuL0FTGshYt7HVJ8P                 S.O.S.  0LcJLqbBmaGUft1e9Mm8HV   
6  0lqYRbRiGc5oiCUW51JH5s              Beautiful  0z4gvV4rjIZ9wHck67ucSV   
7  23yMCTe2AnyIePehtF0Pty            I'm So Paid  0z4gvV4rjIZ9wHck67ucSV   
8  2VqAEP1yY5T523K7bZtL8a   Right Now (Na Na Na)  0z4gvV4rjIZ9wHck67ucSV   
9  0O3vchDpKl6jbS8hvnd2fD           Troublemaker  0z4gvV4rjIZ9wHck67ucSV   

    artist_name                album_id album_name release_date  \
0  3 Doors Down  6aQozyBmD8Edzid1TN7bZA        NaN 

In [9]:
base_with_windows = final_base_table.copy()

base_with_windows["baseline_start"] = base_with_windows["release_date"] - pd.Timedelta(days=30)
base_with_windows["baseline_end"] = base_with_windows["release_date"] - pd.Timedelta(days=7)

base_with_windows["release_window_start"] = base_with_windows["release_date"] - pd.Timedelta(days=7)
base_with_windows["release_window_end"] = base_with_windows["release_date"] + pd.Timedelta(days=7)

base_with_windows.to_csv("base_song_table_with_windows.csv", index=False)

print(base_with_windows.head())

                  song_id              song_name               artist_id  \
0  43Tzmi4Lw0GcZRAE4xqwOg        Citizen/Soldier  2RTUTCvo6onsAnheUk3aL9   
1  3DelzIn0lPwX4PeOJk9zDF       It's Not My Time  2RTUTCvo6onsAnheUk3aL9   
2  4lyphUu7lt4KKuf8jZKjjh                 Get Up  3q7HBObVc0L8jNeTe5Gofh   
3  45EaIq9lKyoQxxTjFkiGbl  Does Your Mother Know  0LcJLqbBmaGUft1e9Mm8HV   
4  29FNeqjOV2kPWGS55qhtGB    Money, Money, Money  0LcJLqbBmaGUft1e9Mm8HV   

    artist_name                album_id album_name release_date  \
0  3 Doors Down  6aQozyBmD8Edzid1TN7bZA        NaN   2008-01-01   
1  3 Doors Down  6aQozyBmD8Edzid1TN7bZA        NaN   2008-01-01   
2       50 Cent  3pjrnGWFZvForSA7Q2J8uA        NaN   2008-01-01   
3          ABBA  2cKZfaz7GiGtZEeQNj1RyR        NaN   2008-01-01   
4          ABBA  2cKZfaz7GiGtZEeQNj1RyR        NaN   2008-01-01   

  release_date_precision baseline_start baseline_end release_window_start  \
0                    day     2007-12-02   2007-12-25           